In [3]:
# !pip3 install spacy 
# !pip3 install scikit-learn
!python3 -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip3 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [47]:
import csv
import spacy
import joblib
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

nlp = spacy.load("en_core_web_sm")

def load_csv_to_tuples(filepath):
    data = []
    try:
        with open(filepath, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            next(reader, None) 
            for i, row in enumerate(reader):
                if len(row) >= 2: 
                    question, label = row[0], row[1].strip()
                    data.append((question, label))
                else:
                    print(f"Warning: Row {i+1} in {filepath} has insufficient elements; Skipping.")
        return data
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return data
    except Exception as e:
        print(f"An error occurred while processing {filepath}: {e}")
        return data

def extract_features(text):
    doc = nlp(text)
    features = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return " ".join(features)

data_files = ["agriculture_query_data.csv", "automotive_query_data.csv", "truck_query_data.csv"]
query_data = []
for data_file in data_files:
    file_data = load_csv_to_tuples(data_file)
    if file_data:
        query_data.extend(file_data)
    else:
        print(f"Skipping {data_file} due to errors during loading.")


texts, labels = zip(*query_data)

cleaned_data = [(text, label) for text, label in zip(texts, labels) if len(text)>0 and len(label)>0]
texts, labels = zip(*cleaned_data)

X = [extract_features(text) for text in texts]
y = list(labels) #Convert labels back to a list

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

 Agriculture       1.00      0.77      0.87        13
        Auto       0.83      1.00      0.91        15
       Truck       1.00      1.00      1.00         8

    accuracy                           0.92        36
   macro avg       0.94      0.92      0.93        36
weighted avg       0.93      0.92      0.92        36



In [44]:

# queries = ["What is a level 1 automotive manufacturer?", "I'm looking for farm equipment.", "My tractor needs repair."]
queries = X_test
for new_query in queries:
    new_query_vec = vectorizer.transform([extract_features(new_query)])
    prediction = model.predict(new_query_vec)[0]
    print(f"Query: {new_query}\nClassification: {prediction}")

Query: specific term contract wheat farmer buyer like flour miller
Classification: Agriculture
Query: fluctuate exchange rate affect competitiveness canadian auto part manufacturer
Classification: Auto
Query: technological innovation impact local specialized freight trucking industry
Classification: Truck
Query: government assistance program available corn farmer Canada
Classification: Agriculture
Query: specific term contract battery manufacturer like Northvolt automotive company
Classification: Auto
Query: wheat farm affect recent cyber security attack
Classification: Agriculture
Query: Aurora prepare customer regulator commercial launch driverless truck
Classification: Auto
Query: innovation product service long distance specialized freight trucking industry
Classification: Truck
Query: type electric vehicle GreenPower manufacture
Classification: Auto
Query: local specialized freight trucking business adapt increase focus sustainability environmental regulation
Classification: Truck

### Save the Model and Vectorizer

In [45]:
model_filename = "classification_model.pkl"
vectorizer_filename = "tfidf_vectorizer.pkl"

joblib.dump(model, model_filename)
joblib.dump(vectorizer, vectorizer_filename)

['tfidf_vectorizer.pkl']

### Test the Model with Test Set

In [46]:
model = joblib.load("classification_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

# queries = ["What is a level 1 automotive manufacturer?", "I'm looking for farm equipment.", "My tractor needs repair."]
queries = X_test

for new_query in queries:
    new_query_vec = vectorizer.transform([extract_features(new_query)])
    prediction = model.predict(new_query_vec)[0]
    print(f"Query: {new_query}\nClassification: {prediction}")


Query: specific term contract wheat farmer buyer like flour miller
Classification: Agriculture
Query: fluctuate exchange rate affect competitiveness canadian auto part manufacturer
Classification: Auto
Query: technological innovation impact local specialized freight trucking industry
Classification: Truck
Query: government assistance program available corn farmer Canada
Classification: Agriculture
Query: specific term contract battery manufacturer like Northvolt automotive company
Classification: Auto
Query: wheat farm affect recent cyber security attack
Classification: Agriculture
Query: Aurora prepare customer regulator commercial launch driverless truck
Classification: Auto
Query: innovation product service long distance specialized freight trucking industry
Classification: Truck
Query: type electric vehicle GreenPower manufacture
Classification: Auto
Query: local specialized freight trucking business adapt increase focus sustainability environmental regulation
Classification: Truck

### Measure Acuracy

In [48]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.9166666666666666
